In [9]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess
from string import Template
from IPython.display import Markdown, display


with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils
import llm_tools as llt

from matplotlib import pyplot as plt


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [10]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
MAX_OUTPUT_TOKENS = 8000
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 4000
REQUESTS_PER_BATCH = 1000
REPLACE = False

In [11]:
rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)


In [12]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [13]:
# Summarize the differences between the clusters

PROMPT = Template("""
In what follows, I will provide you with summaries of agenda items from Los Angeles City Planning Commission meetings. The agenda items have been grouped into three clusters. Several examples are provided from each cluster. Summarize the typical kind of case each cluster contains, and highlight the main differences between the clusters.
                  
$examples
""")

examples_text = ""
for cluster in [0, 1, 2]:
    mask = df['cluster']==cluster
    rows = df.loc[mask].sample(7, random_state=rng).reset_index(drop=True)
    examples_text += "------------------------\n"
    examples_text += f"Cluster: {cluster}\n\n"
    for i, row in rows.iterrows():
        examples_text += f"Example {i+1}:\n"
        examples_text += f"{row['agenda_summary']}\n\n"

prompt = PROMPT.substitute(examples=examples_text)

response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)

display(Markdown(response['response']))


## Cluster Summaries and Comparison

### Cluster 0: Citywide Policy, Code Amendments, and Legislative Actions
These cases involve **city-initiated legislative and administrative actions** that affect policy, zoning codes, or specific plans at a broad (often citywide) level. Typical items include amendments to the Los Angeles Municipal Code, updates to specific plans or overlay districts, adoption of streetscape plans, ordinances authorizing property transactions, and technical corrections to existing legislation. The applicant is almost always the **City of Los Angeles** itself. These items do not involve specific private development projects; rather, they establish or modify the regulatory framework under which development occurs. CEQA review typically relies on exemptions or previously certified EIRs.

### Cluster 1: Large-Scale Private Development Projects (Initial Entitlement Review)
These cases involve **site-specific private development proposals** seeking multiple land use entitlements for substantial mixed-use or residential projects. They typically feature **multi-story buildings (5–13 stories) with dozens to hundreds of units**, affordable housing set-asides, and complex entitlement packages including Density Bonus requests, off-menu incentives, waivers of development standards (FAR, height, setbacks, open space), Conditional Use Permits (often for alcohol or entertainment), Zone Changes, General Plan Amendments, and Site Plan Reviews. The applicants are private developers or LLCs. CEQA review ranges from Categorical Exemptions to Sustainable Communities Environmental Assessments or addenda to prior EIRs. These cases come before the Commission as **primary entitlement decisions** (sometimes with appeals of Director-level approvals).

### Cluster 2: Appeals of Director/Advisory Agency Approvals (TOC Projects and Subdivision Maps)
These cases are **appeals** of lower-level administrative approvals—primarily **Transit Oriented Communities (TOC) incentive program** determinations by the Planning Director and **Vesting Tentative Tract Map** approvals by the Deputy Advisory Agency. The TOC projects tend to be **smaller-to-mid-scale residential buildings** (typically 16–70 units, 6–8 stories) that leverage proximity to transit stops to obtain density, height, FAR, and setback bonuses with affordable housing set-asides for Extremely Low Income households. The tract map cases involve lot mergers, re-subdivisions, and parcel adjustments tied to development projects. The Commission's role here is **appellate**—reviewing decisions already made by staff or advisory bodies.

---

### Key Differences

| Dimension | Cluster 0 | Cluster 1 | Cluster 2 |
|---|---|---|---|
| **Nature of action** | Legislative/policy/code amendments | Site-specific development entitlements | Appeals of prior administrative decisions |
| **Applicant** | City of Los Angeles | Private developers | Private developers (with third-party or project appellants) |
| **Geographic scope** | Citywide or district-wide | Individual project sites | Individual project sites |
| **Commission role** | Recommend adoption to City Council | Primary decision-maker on entitlements | Appellate review body |
| **Project scale** | N/A (no physical development) | Large and complex (100–450+ units, mixed-use) | Smaller to moderate (16–70 units, or subdivision maps) |
| **Primary program/mechanism** | Municipal Code, Specific Plans, Overlay Districts | Density Bonus, CUPs, Zone Changes, GPAs | TOC Incentive Program, Tract/Parcel Maps |
| **CEQA approach** | Exemptions or prior EIRs | Full range (CE, SCEA, Addenda, EIRs) | Typically Categorical Exemptions or prior clearances |

In [14]:
text = r"""Cluster 0, the smallest cluster, consists mainly of non-development actions, such as citywide code amendments, plan updates, or conditional use permits on existing properties. 
Cluster 1 consists primarily of large 
scale private development projects in which the CPC is the initial approver. 
Cluster 2 consists primarily of appeals of decisions made by lower level agencies."""

with open(os.path.join(LOCAL_PATH, "results/cluster_summaries.tex"), 'w') as f:
    f.write(text)

In [15]:
# Summarize the differences between the clusters

PROMPT = Template("""
In what follows, I will provide you with summaries of agenda items from Los Angeles City Planning Commission meetings. The first example has been identified via semantic embeddings as being especially unique or an outlier relative to the other examples. Please describe what makes this example unique compared to the others.
                  
$examples
""")

output_text = ""
for cluster in [0, 1, 2]:
    output_text += "------------------------\n"
    output_text += f"# Cluster: {cluster}\n\n"
    mask = df['cluster']==cluster
    regular_examples = df.loc[mask].sample(7, random_state=rng).reset_index(drop=True)
    mask2 = (df['cluster']==cluster) 
    unique_example = df.loc[mask2].sort_values(by='mahalanobis', ascending=False).head(1).reset_index(drop=True).iloc[0]
    examples_text = f"## Unique Example:\n\n"
    examples_text += f"{unique_example['agenda_summary']}\n\n"
    output_text += f"## Unique Example:\n\n"
    output_text += f"{unique_example['date']} - {unique_example['item_no']}\n\n" 
    output_text += f"{unique_example['agenda_summary']}\n\n"
    for i, row in regular_examples.iterrows():
        examples_text += f"## Regular Example {i+1}:\n\n"
        examples_text += f"{row['agenda_summary']}\n\n"
    prompt = PROMPT.substitute(examples=examples_text)
    response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER, max_output_tokens=MAX_OUTPUT_TOKENS)
    output_text += "# LLM Response:\n\n"
    output_text += f"{response['response']}\n\n"

display(Markdown(output_text))


------------------------
# Cluster: 0

## Unique Example:

2020-08-13 - 5a

The applicant, Hollywood Forever, Inc., seeks approval for the continued operations of an existing 53-acre cemetery in Hollywood, including up to 150 annual special, community, and cultural events (such as charitable fundraisers, film screenings, concerts, and cultural festivals) with alcohol service permitted between 11:00 a.m. and 2:00 a.m. daily. No new permanent structures are proposed. Requested actions include a CEQA categorical exemption, a Conditional Use to allow ancillary special events (up to 150 per year) in the A1 Zone, and a Conditional Use to permit the sale and dispensing of full-line alcoholic beverages for on-site consumption under a Type 88 ABC license.

# LLM Response:

## What Makes the Unique Example Stand Out

The Hollywood Forever Cemetery example is distinctive from the other examples in several notable ways:

### 1. **Unusual Juxtaposition of Land Use and Activity**
The most striking feature is the proposal to host **up to 150 special events per year—including concerts, film screenings, and cultural festivals with alcohol service until 2:00 a.m.—within an active, operating cemetery**. This is a fundamentally unusual pairing. None of the other examples involve repurposing or layering entertainment/nightlife functions onto a land use that is traditionally associated with solemnity, quiet, and permanence. The other examples involve more conventional planning matters: recycling centers, housing redevelopment, code amendments, street realignments, sign modifications, and development agreement terminations.

### 2. **No Physical Development Proposed**
While most of the other examples involve construction, demolition, infrastructure changes, or code/zoning amendments that alter the built environment (new housing units, street widenings, building conversions, sign modifications), the Hollywood Forever case explicitly states **no new permanent structures are proposed**. The entire request is about regulating *activities* on an existing site rather than changing its physical form. This makes it more of a pure operational/use permit case.

### 3. **Scale and Frequency of Events on a Single Parcel**
Requesting permission for **150 events per year** with full alcohol service is an extraordinarily high volume of event programming for any single site, let alone a cemetery. This essentially transforms the cemetery into a part-time entertainment venue—roughly one event every 2.4 days. None of the other examples involve anything approaching this intensity of recurring programmed activity.

### 4. **Alcohol Licensing in an Atypical Context**
The request for a **Type 88 ABC license** (a special event-oriented full-line alcohol license) within an **A1 (Agricultural) Zone** on cemetery grounds is highly unusual. The other examples that touch on commercial activity involve zones and uses where such activity is more expected (M2 industrial zones, commercial districts, specific plan areas).

### 5. **Cultural/Community Dimension**
The application frames the events as charitable, cultural, and community-oriented (fundraisers, cultural festivals), giving it a distinctive civic and cultural character. The other examples are more straightforwardly developmental, regulatory, or infrastructural in nature.

In essence, this example is an outlier because it asks the planning commission to sanction a **high-frequency entertainment and alcohol operation within a 53-acre active cemetery**—a combination of use, context, and scale that is genuinely anomalous in the landscape of typical urban planning decisions.

------------------------
# Cluster: 1

## Unique Example:

2023-10-12 - 10

The project involves construction of a new seven-story, 143.5-foot-tall multi-discipline research facility (USC Discovery and Translation Hub) on the westerly portion of the USC Health Sciences Campus. The building would include two subterranean levels and approximately 201,292 square feet of floor area to support over 84 researchers in both wet and dry laboratory research. Requested actions include a CEQA finding based on a fifth Addendum to a previously certified EIR, and a Major Conditional Use Permit to allow 201,292 square feet of nonresidential floor area in the C2 Zone.

# LLM Response:

## What Makes the Unique Example Distinctive

The unique example stands apart from all the regular examples in several fundamental ways:

### 1. **Institutional/Research Use vs. Residential/Mixed-Use**
This is the most striking difference. The unique example involves a **university research facility** (USC Discovery and Translation Hub) designed for scientific laboratory work supporting 84+ researchers. Every single regular example involves **residential development** — either purely multi-family housing or mixed-use buildings with dwelling units and ground-floor commercial/retail. The unique example contains zero residential units.

### 2. **No Affordable Housing or Density Bonus Framework**
All seven regular examples engage with Los Angeles's **Density Bonus/Affordable Housing** system — they set aside units for Very Low Income or Low Income households and leverage that commitment to obtain incentives like increased FAR, height bonuses, and setback reductions. The unique example has no affordable housing component whatsoever, because it isn't housing. Its entitlement pathway is entirely different: a **Major Conditional Use Permit** for nonresidential floor area in the C2 Zone.

### 3. **CEQA Approach: Addendum to a Prior EIR vs. Categorical Exemptions**
The regular examples almost universally rely on **CEQA categorical exemptions** (Class 32 or similar), which are streamlined findings for smaller-impact projects. The unique example relies on a **fifth Addendum to a previously certified Environmental Impact Report** — indicating it is part of a larger, long-term campus development program with a more complex and layered environmental review history. This suggests a fundamentally different scale of environmental analysis.

### 4. **Institutional Campus Context**
The project is situated on the **USC Health Sciences Campus**, meaning it operates within a master-planned institutional setting rather than on typical urban infill lots. The regular examples all involve standard city parcels — demolishing existing houses, commercial buildings, parking lots, or truck rental facilities to build new urban infill.

### 5. **Building Scale and Program**
At **201,292 square feet** with specialized wet and dry laboratories and two subterranean levels, the building's program is far more complex and specialized than the regular examples, which range from roughly 11,000 to 124,000 square feet of relatively standard residential/commercial construction. The **143.5-foot height** also exceeds all regular examples (which top out around 83–91 feet).

### In Summary
The unique example represents a completely different **project typology** — a large-scale institutional research building on a university campus, processed through a major conditional use permit and a complex CEQA addendum history — in a dataset otherwise entirely composed of urban infill residential and mixed-use developments navigating the city's density bonus and affordable housing incentive framework.

------------------------
# Cluster: 2

## Unique Example:

2023-08-24 - 4

This item concerns action consistent with the terms of a court-issued Writ in Hi Point Neighbor's Association v. City of Los Angeles. The Revised Peremptory Writ of Mandamus, issued May 25, 2023, orders the City to set aside its Tier 3 TOC Affordable Housing Incentive Program approval (DIR-2020-2067-TOC) for a proposed 20-unit multi-family residential building at 1447 South Hi Point Street in the Wilshire Plan Area. The Writ does not limit or control the discretion vested in the City.

# LLM Response:

## What Makes the Unique Example Distinct

The unique example stands apart from all the regular examples in several fundamental ways:

### 1. **It's a Court-Ordered Action, Not an Appeal or Project Approval**
Every regular example involves the Planning Commission hearing an appeal of a planning decision (Zoning Administrator's Adjustment, Director of Planning's approval, Advisory Agency's tract map approval, etc.). The unique example, by contrast, involves the Commission responding to a **Peremptory Writ of Mandamus** — a court order compelling the City to take specific action. This is a fundamentally different procedural posture: the Commission is not exercising its appellate review function but rather complying with a judicial directive.

### 2. **It Involves Undoing a Previous Approval Rather Than Reviewing One**
The regular examples all involve the Commission evaluating whether to uphold, modify, or reverse a planning decision on its merits. The unique example requires the City to **set aside** an approval that was already granted (DIR-2020-2067-TOC). The action is backward-looking and remedial rather than forward-looking and evaluative.

### 3. **The Initiating Party is a Court, Not an Appellant or Applicant**
In every regular example, the proceeding is initiated by a private party (neighborhood groups, unions, environmental organizations, individuals) filing an appeal through the administrative process. Here, the matter comes before the Commission because of **litigation** — a lawsuit (Hi Point Neighbor's Association v. City of Los Angeles) that resulted in a court ruling against the City.

### 4. **No Substantive Project Details Are Under Review**
The regular examples all contain detailed descriptions of proposed developments — building heights, unit counts, FAR ratios, parking configurations, income-restricted units, etc. The unique example mentions only the bare minimum about the project (a 20-unit building) because the substance of the item is the **legal compliance mechanism**, not the project's planning merits.

### 5. **The Legal Framing is Distinctive**
The note that "the Writ does not limit or control the discretion vested in the City" is a carefully worded legal caveat suggesting the City retains authority to re-approve the project through a corrected process. This kind of judicial/administrative interplay language is entirely absent from the regular examples, which operate within standard administrative land-use procedures.

In essence, this item represents the rare intersection of **judicial oversight and planning administration**, whereas all the other items represent routine (if sometimes contentious) administrative land-use review.



In [16]:
text = r"""In cluster 0, the most atypical case was an application for a conditional use permit to operate cultural events such as charitable fundraisers, film screenings, and concerts, and to sell alcoholic beverages on an existing 53-acre cemetery. 
In cluster 1, the most atypical case was an application to permit development of a 143.5 foot tall, 201,292 square foot multi-disciplinary research facility on the USC Health Sciences Campus.
In cluster 2, the most atypical case involved a Writ of Mandamus issued by the Los Angeles Superior Court ordering the City to set aside a previous approval on a 20-unit housing development because it hadn't adequately demonstrated TOC Tier 3 eligibility. However, the Writ did not limit the discretion vested in the City as to how to proceed with the project."""

with open(os.path.join(LOCAL_PATH, "results/unique_examples.tex"), 'w') as f:
    f.write(text)
